In [16]:
import pickle

In [17]:
path = "/path/to/additional_evaluations/codebook_util/eval/F16_K8192_Z32_S16/codebook_statistics/1000000.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

print(len(data))

16


In [25]:
path = "/path/to/cluster/output/eval/F16_K128_Z4_S2/codebook_statistics/500000.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

print(len(data))
print(data)

128
[CodebookUsageEntry(codebook_index=0, bincount=tensor([165446.,  19641., 118367., 117368.,  69202.,  60262., 146384.,  77982.,
         32078., 161239., 116827., 111795.,  97471.,  77427.,  61892.,  84795.,
        129578., 131806., 111545.,  94385.,  19683., 165722., 133079.,  63139.,
         38338.,  94983.,  39648.,  68450., 141971., 102787., 154122.,  32551.,
         65812., 113179.,  62100.,  21569., 131645., 149691.,  66489., 154321.,
         48296.,  24269., 149712., 136626., 126623., 163091., 119194.,  33612.,
        170466.,  23605., 116447., 127046.,  81939., 142415.,  75862., 142471.,
         51087.,  76103., 123717.,  91577., 164216., 102010.,  62886.,  56579.,
         29163., 136854., 135506.,  96149.,  93551.,  33295., 147544., 123758.,
        157361., 152274.,  55843., 138055., 137548., 136812.,  88965.,  96281.,
        157915., 107280.,   8040., 123654., 110637., 161876., 158021., 100181.,
         39160., 141477.,  81547.,  54640.,  60177.,  34962.,  13426.

In [2]:
path = "/path/to/tmp/test_eval/eval/F16_K128_Z256_S128/codebook_statistics/1200000.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

print(data)

[CodebookUsageEntry(codebook_index=0, bincount=tensor([7.6100e+02, 2.7900e+02, 0.0000e+00, 1.5000e+02, 3.2000e+02, 4.5200e+02,
        2.3400e+02, 3.4000e+01, 1.1740e+03, 1.0000e+00, 3.8700e+02, 0.0000e+00,
        1.2340e+03, 1.4280e+03, 1.4200e+02, 2.0900e+02, 0.0000e+00, 1.2030e+03,
        1.1870e+03, 1.4020e+03, 7.9500e+02, 3.1900e+02, 5.5200e+02, 2.5300e+02,
        8.7000e+01, 1.0710e+03, 8.3500e+02, 2.0000e+00, 4.5400e+02, 0.0000e+00,
        1.0630e+03, 9.7400e+02, 0.0000e+00, 9.2300e+02, 2.2900e+02, 2.1000e+02,
        4.5600e+02, 4.3600e+02, 0.0000e+00, 7.7000e+02, 1.6100e+02, 3.6900e+02,
        0.0000e+00, 3.2200e+02, 7.6400e+02, 1.5700e+02, 1.3700e+02, 1.0880e+03,
        3.3000e+02, 8.0000e+00, 3.0000e+01, 7.1000e+02, 7.8600e+02, 4.5900e+02,
        8.2400e+02, 1.2540e+03, 3.4200e+02, 3.8000e+02, 1.4750e+03, 1.0980e+03,
        8.5100e+02, 1.1860e+03, 5.0000e+00, 9.7600e+02, 1.0250e+03, 7.9000e+01,
        1.4000e+03, 1.2700e+02, 5.5000e+02, 2.4100e+02, 1.1700e+02, 4.060

In [ ]:
import os
import pickle
from dataclasses import dataclass
from typing import List
from vv.shared.shared_types import CodebookUsage, CodebookUsageEntry
from tqdm import tqdm

def convert_old_to_new_format(old_data: dict) -> CodebookUsage:
    new_data: CodebookUsage = []
    for key, value in old_data.items():
        # Extract the space number from the key
        space_number = int(key.split('_')[-1])
        # Extract the tensor from the Parameter object
        tensor_value = value.data if hasattr(value, 'data') else value
        # Create a new CodebookEntry
        new_data.append(CodebookUsageEntry(codebook_index=space_number, bincount=tensor_value))
    return new_data

def process_folder(root_folder: str):
    # Count the total number of directories and files
    total_items = sum(len(dirnames) for _, dirnames, _ in os.walk(root_folder))
    
    # Use tqdm with the total count
    with tqdm(total=total_items, desc="Processing folders and files") as pbar:
        for dirpath, dirnames, filenames in os.walk(root_folder):
            # Rename "codebook_utilization" to "codebook_statistics"
            if "codebook_utilization" in dirnames:
                old_path = os.path.join(dirpath, "codebook_utilization")
                new_path = os.path.join(dirpath, "codebook_statistics")
                os.rename(old_path, new_path)
                dirnames.remove("codebook_utilization")
                dirnames.append("codebook_statistics")
                pbar.update(1)  # Update progress bar for the renamed folder

            # Process .pkl files in "codebook_statistics" folders
            if "codebook_statistics" in dirpath:
                for filename in filenames:
                    if filename.endswith(".pkl"):
                        file_path = os.path.join(dirpath, filename)
                        with open(file_path, "rb") as f:
                            old_data = pickle.load(f)
                        
                        # Convert the data to the new format
                        new_data = convert_old_to_new_format(old_data)
                        
                        # Save the new data back to the same file
                        with open(file_path, "wb") as f:
                            pickle.dump(new_data, f)
                        pbar.update(1)  # Update progress bar for each processed file

In [22]:
path = "/path/to/cluster/output/eval"

process_folder(path)

Processing folders and files:  52%|█████▏    | 800/1539 [00:56<00:52, 14.20it/s]


In [31]:
import pandas as pd

def check_duplicates_in_results(root_folder: str):
    duplicates = []  # List to store paths of files with duplicate step numbers

    # Count the total number of directories and files
    total_items = sum(len(dirnames) + len(filenames) for _, dirnames, filenames in os.walk(root_folder))

    # Use tqdm to show progress
    with tqdm(total=total_items, desc="Checking for duplicates in results.txt") as pbar:
        for dirpath, dirnames, filenames in os.walk(root_folder):
            for filename in filenames:
                if filename == "results.txt":  # Check for the results.txt file
                    file_path = os.path.join(dirpath, filename)
                    try:
                        # Load the CSV file
                        df = pd.read_csv(file_path)

                        # Check for duplicate step numbers
                        if "step" in df.columns:  # Ensure the 'step' column exists
                            if df["step"].duplicated().any():
                                duplicates.append(file_path)
                    except Exception as e:
                        print(f"Error processing {file_path}: {e}")

                    pbar.update(1)  # Update progress bar for each file

    # Print the list of files with duplicate step numbers
    if duplicates:
        print("Files with duplicate step numbers:")
        for file in duplicates:
            print(file)
    else:
        print("No duplicates found in any results.txt files.")

In [32]:
root_folder = "/path/to/cluster/output/eval"
check_duplicates_in_results(root_folder)

Checking for duplicates in results.txt:   8%|▊         | 125/1530 [00:00<00:03, 361.57it/s]

No duplicates found in any results.txt files.


In [36]:
from collections import Counter

def find_and_check_duplicate_folders(root_folder: str):
    # List to store paths of folders starting with 'F'
    folder_paths = []

    # Traverse the directory structure
    for dirpath, dirnames, _ in tqdm(os.walk(root_folder), desc="Scanning folders"):
        for dirname in dirnames:
            if dirname.startswith("F"):  # Check if folder name starts with 'F'
                folder_paths.append(os.path.join(dirpath, dirname))

    # Extract folder names from the paths
    folder_names = [os.path.basename(path) for path in folder_paths]

    # Count occurrences of each folder name
    folder_name_counts = Counter(folder_names)

    # Find duplicates
    duplicates = [name for name, count in folder_name_counts.items() if count > 1]

    # Print results
    if duplicates:
        print("Duplicate folder names found:")
        for duplicate in duplicates:
            print(f"Folder name: {duplicate}")
            print("Paths:")
            for path in folder_paths:
                if os.path.basename(path) == duplicate:
                    print(f"  {path}")
    else:
        print("No duplicate folder names found.")

In [43]:
root_folder = "/path/to/models"
find_and_check_duplicate_folders(root_folder)

Scanning folders: 2804it [00:01, 2490.59it/s]

No duplicate folder names found.


In [106]:
def find_highest_version_and_config(root_folder: str):
    results = []  # List to store tuples of (model_name, highest_version_ckpt_path, config_path)

    # Traverse the directory structure
    for dirpath, dirnames, filenames in tqdm(os.walk(root_folder), desc="Scanning models"):
        # Check if we are in a model directory (contains version_<number> subdirectories)
        model_name = os.path.basename(dirpath)
        version_dirs = [d for d in dirnames if d.startswith("version_")]

        if version_dirs:
            # Extract version numbers and find the highest version
            version_numbers = [int(d.split("_")[1]) for d in version_dirs]
            highest_version = max(version_numbers)
            highest_version_dir = f"version_{highest_version}"

            # Path to the highest version directory
            highest_version_path = os.path.join(dirpath, highest_version_dir)

            # Check if config.yaml exists in the highest version directory
            config_path = os.path.join(highest_version_path, "config.yaml")
            if not os.path.exists(config_path):
                print(f"Warning: config.yaml not found in {highest_version_path}")
                continue

            # Check if the "checkpoints" directory exists
            highest_version_path_checkpoint = os.path.join(highest_version_path, "checkpoints")
            if not os.path.exists(highest_version_path_checkpoint):
                print(f"Warning: 'checkpoints' directory not found in {highest_version_path}")
                continue
            

            # Find the step subdirectories in the highest version directory
            step_dirs = [d for d in os.listdir(highest_version_path_checkpoint) if os.path.isdir(os.path.join(highest_version_path_checkpoint, d))]
            #step_dirs = [os.path.join(highest_version_path_checkpoint, d) for d in os.listdir(highest_version_path_checkpoint)]
            # Extract the step numbers from the step directories
            step_dirs_with_steps = [(d, int(d.split("=")[-1].split("-")[0])) for d in step_dirs if "step=" in d]
            
            # Find the directory with the highest step number
            if step_dirs_with_steps:
                highest_step_dir = max(step_dirs_with_steps, key=lambda x: x[1])[0]
                step_dirs = [os.path.join(highest_version_path_checkpoint, highest_step_dir)]

            # Look for .ckpt files in the step subdirectories
            ckpt_files = []
            for step_path in step_dirs:
                ckpt_files.extend([os.path.join(step_path, f) for f in os.listdir(step_path) if f.endswith(".ckpt")])

            if ckpt_files:
                # Save the first .ckpt file found (or modify logic if needed)
                highest_version_ckpt_path = ckpt_files[0]
                results.append((model_name, highest_version_ckpt_path, config_path))

    return results

In [108]:
root_folder = "/path/to/models"
results = find_highest_version_and_config(root_folder)

print(results[0])

Scanning models: 1412it [00:00, 2161.01it/s]

Scanning models: 2769it [00:01, 2189.18it/s]

('F16_K128_Z16_S16', '/path/to/models/Z16/F16_K128_Z16_S16/version_9/checkpoints/epoch=11-step=1000000-val/total_loss=0.3380.ckpt', '/path/to/models/Z16/F16_K128_Z16_S16/version_9/config.yaml')


In [113]:
import os
import shutil

def copy_checkpoints_to_new_directory(results, destination_folder):
    """
    Copies model checkpoints to a new directory with a structured hierarchy.

    Args:
        results (list): List of tuples (model_name, highest_version_ckpt_path, config_path).
        destination_folder (str): Path to the destination directory.
    """
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    for model_name, ckpt_path, config_path in results:
        # Extract version and step information from the checkpoint path
        version = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(ckpt_path))))  # e.g., version_0
        checkpoints = os.path.basename(os.path.dirname(os.path.dirname(ckpt_path)))  # e.g., checkpoints
        step = os.path.basename(os.path.dirname(ckpt_path))  # e.g., epoch=11-step=1000000-val
        ckpt_filename = os.path.basename(ckpt_path)  # e.g., total_loss=0.3380.ckpt

        # Create the new directory structure
        model_dir = os.path.join(destination_folder, model_name)
        version_dir = os.path.join(model_dir, version)
        checkpoints_dir = os.path.join(version_dir, checkpoints, step)

        # Ensure the directories exist
        os.makedirs(checkpoints_dir, exist_ok=True)

        # Copy the checkpoint file
        destination_ckpt_path = os.path.join(checkpoints_dir, ckpt_filename)
        shutil.copy2(ckpt_path, destination_ckpt_path)

        # Copy the config.yaml file to the version directory
        destination_config_path = os.path.join(version_dir, "config.yaml")
        shutil.copy2(config_path, destination_config_path)

        print(f"Copied {ckpt_path} to {destination_ckpt_path}")
        print(f"Copied {config_path} to {destination_config_path}")



In [115]:
# Example usage
destination_folder = "/path/to/additional_evaluations/codebook_util/models"
copy_checkpoints_to_new_directory(results, destination_folder)

Copied /path/to/models/Z16/F16_K128_Z16_S16/version_9/checkpoints/epoch=11-step=1000000-val/total_loss=0.3380.ckpt to /path/to/additional_evaluations/codebook_util/models/F16_K128_Z16_S16/version_9/checkpoints/epoch=11-step=1000000-val/total_loss=0.3380.ckpt
Copied /path/to/models/Z16/F16_K128_Z16_S16/version_9/config.yaml to /path/to/additional_evaluations/codebook_util/models/F16_K128_Z16_S16/version_9/config.yaml
Copied /path/to/models/Z16/F16_K16384_Z16_S16/version_9/checkpoints/epoch=11-step=1000000-val/total_loss=0.3350.ckpt to /path/to/additional_evaluations/codebook_util/models/F16_K16384_Z16_S16/version_9/checkpoints/epoch=11-step=1000000-val/total_loss=0.3350.ckpt
Copied /path/to/models/Z16/F16_K16384_Z16_S16/version_9/config.yaml to /path/to/additional_evaluations/codebook_util/models/F16_K16384_Z16_S16/version_9/config.yaml
Copied /path/to/models/Z16/F16_K8192_Z16_S8/version_5/checkpoints/epoch=7-step=600000-val/total_loss=0.3595.ckpt to /path/to/additional_evaluations/code

In [5]:
from vv.shared.shared_types import CodebookUsage, CodebookUsageEntry
from vv.evaluation.core.util import codebook_statistics_summary
import torch

In [15]:
test = [
    CodebookUsageEntry(codebook_index=0, bincount=torch.Tensor([10, 10000, 10]))
]
print(codebook_statistics_summary(test))

{'entropy': 0.015785910189151764, 'normalized_entropy': 0.014368954114615917, 'perplexity': 1.0159112215042114, 'normalized_perplexity': 0.3386370837688446, 'utilization_per_subcodebook': [1.0], 'avg_utilization': 1.0}
